# Create Three Continual-Learning Stage Files

Selected stages:

- Stage 1: 2010–2018
- Stage 2: 2019–2022
- Stage 3: 2023

For every stage:

1. A chronological holdout is taken from the beginning.
2. A 24-hour NF-only gap separates the holdout from training.
3. Holdout samples are not augmented or used for training.
4. Training and holdout files contain only:
   - `label`
   - `goes_class`

NF-only gaps are also removed between consecutive stages.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository root.
possible_roots = [Path.cwd(), *Path.cwd().parents]

ROOT = next(
    path for path in possible_roots
    if (path / "data_labeling").is_dir()
)

LABEL_DIR = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "simplified_data_labels"
)

OLD_LABEL_FILE = LABEL_DIR / "labels_2010_2018_binary.csv"
NEW_LABEL_FILE = LABEL_DIR / "labels_2019_2026_july_binary.csv"

# Use a new directory so the earlier stage files are preserved.
OUTPUT_DIR = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "continual_stages_2010_2023_candidate_A"
)

print("Repository:", ROOT)
print("Output directory:", OUTPUT_DIR)

assert OLD_LABEL_FILE.is_file()
assert NEW_LABEL_FILE.is_file()

Repository: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Output directory: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A


In [2]:
old_labels = pd.read_csv(OLD_LABEL_FILE)
new_labels = pd.read_csv(NEW_LABEL_FILE)

df = pd.concat(
    [old_labels, new_labels],
    ignore_index=True,
)

In [3]:
df

,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0
...,...,...
128088,2026/07/31/HMI.m2026.07.31_19.00.00.jpg,0
128089,2026/07/31/HMI.m2026.07.31_20.00.00.jpg,0
128090,2026/07/31/HMI.m2026.07.31_21.00.00.jpg,0
128091,2026/07/31/HMI.m2026.07.31_22.00.00.jpg,0


In [4]:
# Convert the target into binary form.
target_mapping = {
    "NF": 0,
    "FL": 1,
    "0": 0,
    "1": 1,
    0: 0,
    1: 1,
}

df["target"] = df["goes_class"].map(target_mapping)

In [5]:
df

,label,goes_class,target
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0,0
...,...,...,...
128088,2026/07/31/HMI.m2026.07.31_19.00.00.jpg,0,0
128089,2026/07/31/HMI.m2026.07.31_20.00.00.jpg,0,0
128090,2026/07/31/HMI.m2026.07.31_21.00.00.jpg,0,0
128091,2026/07/31/HMI.m2026.07.31_22.00.00.jpg,0,0


In [6]:
# Extract timestamps from image paths.
timestamp_parts = df["label"].str.extract(
    r"HMI\.m(?P<date>\d{4}\.\d{2}\.\d{2})_"
    r"(?P<time>\d{2}\.\d{2}\.\d{2})"
)

df["timestamp"] = pd.to_datetime(
    timestamp_parts["date"] + " " + timestamp_parts["time"],
    format="%Y.%m.%d %H.%M.%S",
    errors="coerce",
)

df["year"] = df["timestamp"].dt.year

# Only use 2010–2023 here.
stage_source = (
    df[df["year"].between(2010, 2023)]
    .dropna(subset=["timestamp", "target"])
    .copy()
)

In [7]:
stage_source["target"] = stage_source["target"].astype(int)
stage_source["goes_class"] = stage_source["target"]

stage_source = (
    stage_source
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Rows:", len(stage_source))
print("First timestamp:", stage_source["timestamp"].min())
print("Last timestamp:", stage_source["timestamp"].max())
print("Duplicate paths:", stage_source["label"].duplicated().sum())
print("Duplicate timestamps:", stage_source["timestamp"].duplicated().sum())

Rows: 106093
First timestamp: 2010-12-06 07:00:00
Last timestamp: 2023-12-31 23:00:00
Duplicate paths: 0
Duplicate timestamps: 0


In [8]:
STAGE_PERIODS = {
    1: (2010, 2018),
    2: (2019, 2022),
    3: (2023, 2023),
}

HOLDOUT_FRACTION = 0.15
GAP_HOURS = 24

# Require enough FL images to calculate useful holdout metrics.
MIN_HOLDOUT_FL = 30
MIN_TRAIN_FL = 100

print("Selected stages:")

for stage_number, years in STAGE_PERIODS.items():
    print(
        f"Stage {stage_number}: "
        f"{years[0]}–{years[1]}"
    )

Selected stages:
Stage 1: 2010–2018
Stage 2: 2019–2022
Stage 3: 2023–2023


In [9]:
def inspect_gap(data, gap_start, gap_hours=24):
    """
    Check whether a period contains exactly one image per hour
    and whether every image is NF.
    """

    gap_start = pd.Timestamp(gap_start)
    gap_end = gap_start + pd.Timedelta(
        hours=gap_hours - 1
    )

    expected_times = pd.date_range(
        gap_start,
        gap_end,
        freq="h",
    )

    selected = data[
        data["timestamp"].between(
            gap_start,
            gap_end,
        )
    ].copy()

    actual_times = pd.DatetimeIndex(
        selected["timestamp"].sort_values()
    )

    complete = (
        len(selected) == gap_hours
        and actual_times.equals(expected_times)
    )

    flare_count = int(selected["target"].sum())

    return {
        "gap_start": gap_start,
        "gap_end": gap_end,
        "rows": len(selected),
        "FL": flare_count,
        "NF": len(selected) - flare_count,
        "complete": complete,
        "valid_nf_gap": complete and flare_count == 0,
    }

In [10]:
def find_boundary_gap(data, boundary):
    """
    Find an NF-only 24-hour gap immediately before or
    immediately after a stage boundary.
    """

    boundary = pd.Timestamp(boundary)

    possible_starts = [
        boundary - pd.Timedelta(hours=GAP_HOURS),
        boundary,
    ]

    results = [
        inspect_gap(
            data,
            start,
            gap_hours=GAP_HOURS,
        )
        for start in possible_starts
    ]

    valid_results = [
        result
        for result in results
        if result["valid_nf_gap"]
    ]

    if not valid_results:
        raise RuntimeError(
            f"No complete NF-only {GAP_HOURS}-hour gap "
            f"was found next to {boundary}."
        )

    # Prefer the earlier-side gap when both are valid.
    return valid_results[0]


stage_boundaries = [
    pd.Timestamp("2019-01-01 00:00:00"),
    pd.Timestamp("2023-01-01 00:00:00"),
]

boundary_gaps = [
    find_boundary_gap(stage_source, boundary)
    for boundary in stage_boundaries
]

boundary_gap_table = pd.DataFrame(boundary_gaps)

display(boundary_gap_table)

,gap_start,gap_end,rows,FL,NF,complete,valid_nf_gap
0,2019-01-01,2019-01-01 23:00:00,24,0,24,True,True
1,2022-12-31,2022-12-31 23:00:00,24,0,24,True,True


In [11]:
boundary_gap_mask = pd.Series(
    False,
    index=stage_source.index,
)

for gap in boundary_gaps:
    boundary_gap_mask |= stage_source["timestamp"].between(
        gap["gap_start"],
        gap["gap_end"],
    )

boundary_gap_images = stage_source[
    boundary_gap_mask
].copy()

data_after_boundary_gaps = stage_source[
    ~boundary_gap_mask
].copy()

print("Images removed between stages:", len(boundary_gap_images))
print(
    "FL images removed between stages:",
    boundary_gap_images["target"].sum(),
)
print(
    "NF images removed between stages:",
    len(boundary_gap_images)
    - boundary_gap_images["target"].sum(),
)

display(
    boundary_gap_images.groupby("year")["target"]
    .agg(rows="size", FL="sum")
)

Images removed between stages: 48
FL images removed between stages: 0
NF images removed between stages: 48


,rows,FL
year,,
2019,24,0
2022,24,0


In [12]:
def find_internal_holdout_gap(
    stage_data,
    holdout_fraction=0.15,
    gap_hours=24,
    minimum_holdout_fl=30,
    minimum_train_fl=100,
):
    """
    Find an NF-only gap near the requested holdout fraction.

    Data before the gap become the holdout.
    Data after the gap become training data.
    """

    stage_data = (
        stage_data
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    hourly_target = (
        stage_data
        .groupby("timestamp")["target"]
        .max()
        .sort_index()
    )

    complete_hours = pd.date_range(
        hourly_target.index.min(),
        hourly_target.index.max(),
        freq="h",
    )

    hourly_target = hourly_target.reindex(
        complete_hours
    )

    rolling_count = hourly_target.rolling(
        gap_hours,
        min_periods=gap_hours,
    ).count()

    rolling_fl = hourly_target.rolling(
        gap_hours,
        min_periods=gap_hours,
    ).sum()

    valid_gap_ends = hourly_target.index[
        (rolling_count == gap_hours)
        & (rolling_fl == 0)
    ]

    total_available = len(stage_data) - gap_hours
    target_holdout_size = round(
        holdout_fraction * total_available
    )

    possible_splits = []

    for gap_end in valid_gap_ends:
        gap_start = (
            gap_end
            - pd.Timedelta(hours=gap_hours - 1)
        )

        holdout = stage_data[
            stage_data["timestamp"] < gap_start
        ]

        train = stage_data[
            stage_data["timestamp"] > gap_end
        ]

        holdout_fl = int(holdout["target"].sum())
        train_fl = int(train["target"].sum())

        if (
            len(holdout) > 0
            and len(train) > 0
            and holdout_fl >= minimum_holdout_fl
            and train_fl >= minimum_train_fl
        ):
            possible_splits.append({
                "gap_start": gap_start,
                "gap_end": gap_end,
                "holdout_rows": len(holdout),
                "train_rows": len(train),
                "holdout_FL": holdout_fl,
                "train_FL": train_fl,
                "distance_from_target": abs(
                    len(holdout) - target_holdout_size
                ),
            })

    if not possible_splits:
        raise RuntimeError(
            "No suitable NF-only holdout gap was found."
        )

    best_split = min(
        possible_splits,
        key=lambda item: item["distance_from_target"],
    )

    holdout = stage_data[
        stage_data["timestamp"]
        < best_split["gap_start"]
    ].copy()

    train = stage_data[
        stage_data["timestamp"]
        > best_split["gap_end"]
    ].copy()

    purged_gap = stage_data[
        stage_data["timestamp"].between(
            best_split["gap_start"],
            best_split["gap_end"],
        )
    ].copy()

    return {
        "train": train,
        "holdout": holdout,
        "gap": purged_gap,
        "gap_start": best_split["gap_start"],
        "gap_end": best_split["gap_end"],
    }

In [13]:
stage_splits = {}

for stage_number, (start_year, end_year) in STAGE_PERIODS.items():
    selected_stage = data_after_boundary_gaps[
        data_after_boundary_gaps["year"].between(
            start_year,
            end_year,
        )
    ].copy()

    split = find_internal_holdout_gap(
        stage_data=selected_stage,
        holdout_fraction=HOLDOUT_FRACTION,
        gap_hours=GAP_HOURS,
        minimum_holdout_fl=MIN_HOLDOUT_FL,
        minimum_train_fl=MIN_TRAIN_FL,
    )

    split["original_rows"] = len(
        stage_source[
            stage_source["year"].between(
                start_year,
                end_year,
            )
        ]
    )

    split["available_after_boundary_gap"] = len(
        selected_stage
    )

    split["years"] = f"{start_year}-{end_year}"

    stage_splits[stage_number] = split

print("Created splits for stages:", list(stage_splits))

Created splits for stages: [1, 2, 3]


In [14]:
summary_rows = []

for stage_number, split in stage_splits.items():
    train = split["train"]
    holdout = split["holdout"]
    gap = split["gap"]

    summary_rows.append({
        "Stage": stage_number,
        "Years": split["years"],
        "Original": split["original_rows"],
        "After boundary gaps": (
            split["available_after_boundary_gap"]
        ),
        "Train": len(train),
        "Train NF": int((train["target"] == 0).sum()),
        "Train FL": int((train["target"] == 1).sum()),
        "Holdout": len(holdout),
        "Holdout NF": int(
            (holdout["target"] == 0).sum()
        ),
        "Holdout FL": int(
            (holdout["target"] == 1).sum()
        ),
        "Internal gap": len(gap),
        "Gap FL": int(gap["target"].sum()),
    })

stage_summary = pd.DataFrame(summary_rows)

display(stage_summary)

,Stage,Years,Original,After boundary gaps,Train,Train NF,Train FL,Holdout,Holdout NF,Holdout FL,Internal gap,Gap FL
0,1,2010-2018,63285,63285,53870,46511,7359,9391,7775,1616,24,0
1,2,2019-2022,34164,34116,22160,19407,2753,11932,11884,48,24,0
2,3,2023-2023,8644,8644,7319,3985,3334,1301,541,760,24,0


In [15]:
boundary_rows = []

for stage_number, split in stage_splits.items():
    train = split["train"]
    holdout = split["holdout"]

    boundary_rows.append({
        "Stage": stage_number,
        "Years": split["years"],
        "Holdout start": holdout["timestamp"].min(),
        "Holdout end": holdout["timestamp"].max(),
        "Gap start": split["gap_start"],
        "Gap end": split["gap_end"],
        "Training start": train["timestamp"].min(),
        "Training end": train["timestamp"].max(),
    })

stage_boundaries_table = pd.DataFrame(
    boundary_rows
)

display(stage_boundaries_table)

,Stage,Years,Holdout start,Holdout end,Gap start,Gap end,Training start,Training end
0,1,2010-2018,2010-12-06 07:00:00,2012-06-26 16:00:00,2012-06-26 17:00:00,2012-06-27 16:00:00,2012-06-27 17:00:00,2018-12-30 23:00:00
1,2,2019-2022,2019-01-02 00:00:00,2020-05-29 07:00:00,2020-05-29 08:00:00,2020-05-30 07:00:00,2020-05-30 08:00:00,2022-12-30 23:00:00
2,3,2023-2023,2023-01-01 00:00:00,2023-02-25 18:00:00,2023-02-25 19:00:00,2023-02-26 18:00:00,2023-02-26 20:00:00,2023-12-31 23:00:00


In [16]:
all_saved_paths = []

for stage_number, split in stage_splits.items():
    train = split["train"]
    holdout = split["holdout"]
    gap = split["gap"]

    # No duplicates inside either split.
    assert not train["label"].duplicated().any()
    assert not holdout["label"].duplicated().any()

    # Train and holdout must not overlap.
    assert set(train["label"]).isdisjoint(
        set(holdout["label"])
    )

    # The internal gap must be complete and NF-only.
    assert len(gap) == GAP_HOURS
    assert gap["target"].sum() == 0

    # Both classes must be present.
    assert train["target"].nunique() == 2
    assert holdout["target"].nunique() == 2

    # Verify chronological separation.
    assert holdout["timestamp"].max() < split["gap_start"]
    assert split["gap_end"] < train["timestamp"].min()

    all_saved_paths.extend(train["label"].tolist())
    all_saved_paths.extend(holdout["label"].tolist())

# No image may appear in more than one saved file.
assert len(all_saved_paths) == len(set(all_saved_paths))

# Inter-stage gaps must contain only NF images.
assert boundary_gap_images["target"].sum() == 0

print("All validations passed.")
print("No train/holdout overlap detected.")
print("No paths are reused between stages.")
print("Every gap contains only NF images.")

All validations passed.
No train/holdout overlap detected.
No paths are reused between stages.
Every gap contains only NF images.


In [17]:
csv_previews = {}

for stage_number, split in stage_splits.items():
    train_csv = (
        split["train"][["label", "goes_class"]]
        .copy()
    )

    holdout_csv = (
        split["holdout"][["label", "goes_class"]]
        .copy()
    )

    train_csv["goes_class"] = (
        train_csv["goes_class"].astype(int)
    )

    holdout_csv["goes_class"] = (
        holdout_csv["goes_class"].astype(int)
    )

    csv_previews[stage_number] = {
        "train": train_csv,
        "holdout": holdout_csv,
    }

    print(f"\nStage {stage_number} train:")
    display(train_csv.head())

    print(f"Stage {stage_number} holdout:")
    display(holdout_csv.head())


Stage 1 train:


,label,goes_class
9415,2012/06/27/HMI.m2012.06.27_17.00.00.jpg,1
9416,2012/06/27/HMI.m2012.06.27_18.00.00.jpg,1
9417,2012/06/27/HMI.m2012.06.27_19.00.00.jpg,1
9418,2012/06/27/HMI.m2012.06.27_20.00.00.jpg,1
9419,2012/06/27/HMI.m2012.06.27_21.00.00.jpg,1


Stage 1 holdout:


,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0



Stage 2 train:


...

,label,goes_class
0,2019/01/02/HMI.m2019.01.02_00.00.00.jpg,0
1,2019/01/02/HMI.m2019.01.02_01.00.00.jpg,0
2,2019/01/02/HMI.m2019.01.02_02.00.00.jpg,0
3,2019/01/02/HMI.m2019.01.02_03.00.00.jpg,0
4,2019/01/02/HMI.m2019.01.02_04.00.00.jpg,0



Stage 3 train:


,label,goes_class
1325,2023/02/26/HMI.m2023.02.26_20.00.00.jpg,0
1326,2023/02/26/HMI.m2023.02.26_21.00.00.jpg,0
1327,2023/02/26/HMI.m2023.02.26_22.00.00.jpg,0
1328,2023/02/26/HMI.m2023.02.26_23.00.00.jpg,0
1329,2023/02/27/HMI.m2023.02.27_00.00.00.jpg,0


Stage 3 holdout:


,label,goes_class
0,2023/01/01/HMI.m2023.01.01_00.00.00.jpg,0
1,2023/01/01/HMI.m2023.01.01_01.00.00.jpg,0
2,2023/01/01/HMI.m2023.01.01_02.00.00.jpg,0
3,2023/01/01/HMI.m2023.01.01_03.00.00.jpg,0
4,2023/01/01/HMI.m2023.01.01_04.00.00.jpg,0


In [18]:
OVERWRITE_EXISTING = False

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

output_files = []

for stage_number, files in csv_previews.items():
    train_path = (
        OUTPUT_DIR
        / f"Stage{stage_number}_train.csv"
    )

    holdout_path = (
        OUTPUT_DIR
        / f"Stage{stage_number}_holdout.csv"
    )

    for path in [train_path, holdout_path]:
        if path.exists() and not OVERWRITE_EXISTING:
            raise FileExistsError(
                f"{path} already exists. "
                "Set OVERWRITE_EXISTING=True only if you "
                "intend to replace it."
            )

    files["train"].to_csv(
        train_path,
        index=False,
    )

    files["holdout"].to_csv(
        holdout_path,
        index=False,
    )

    output_files.extend(
        [train_path, holdout_path]
    )

print("Created files:")

for path in output_files:
    print(path)

Created files:
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage1_train.csv
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage1_holdout.csv
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage2_train.csv
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage2_holdout.csv
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage3_train.csv
/Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A/Stage3_h

In [19]:
verification_rows = []

for path in sorted(output_files):
    saved = pd.read_csv(path)

    assert list(saved.columns) == [
        "label",
        "goes_class",
    ]

    assert saved["goes_class"].isin(
        [0, 1]
    ).all()

    assert not saved["label"].duplicated().any()
    assert not saved.isna().any().any()

    verification_rows.append({
        "File": path.name,
        "Rows": len(saved),
        "NF": int((saved["goes_class"] == 0).sum()),
        "FL": int((saved["goes_class"] == 1).sum()),
        "Duplicates": saved["label"].duplicated().sum(),
        "Missing values": int(saved.isna().sum().sum()),
    })

verification = pd.DataFrame(
    verification_rows
)

display(verification)

print("All six saved files passed verification.")

,File,Rows,NF,FL,Duplicates,Missing values
0,Stage1_holdout.csv,9391,7775,1616,0,0
1,Stage1_train.csv,53870,46511,7359,0,0
2,Stage2_holdout.csv,11932,11884,48,0,0
3,Stage2_train.csv,22160,19407,2753,0,0
4,Stage3_holdout.csv,1301,541,760,0,0
5,Stage3_train.csv,7319,3985,3334,0,0


All six saved files passed verification.
